# Native oblivious amplitude amplification

Keep the source's seed-one random four-amplitude input and both experiments. The equal-weight XX/XZ/ZX/ZZ block encoding is one half of the unitary H tensor H. A single block-only amplification iteration raises the good-block probability from one quarter to one. The input state preparation is not included in the amplification reflection.

This Notebook uses the existing native qfunc helpers and public named probability readout. No cloud execution, source importer or source-language code is required.


In [ ]:
import numpy as np

import flagquantum as fq
from flagquantum import synth
from examples.synthesis.algorithms.amplitude_amplification_and_estimation.oblivious_amplitude_amplification.overview.main import (
    Stage,
    source_amplitudes,
)

amplitudes = source_amplitudes()
print("Seed-one normalized input:", amplitudes)


## Reversible quantum composition

The helper `encode` prepares and unprepares the Pauli selector. The oracle acts only on `block == 0`; `reflect` composes the checked adjoint and within/apply reflection, retaining its global sign. The compiler owns physical wires and reversible work.


In [ ]:
from flagquantum.synth import (
    Output,
    QArray,
    QBit,
    allocate,
    inplace_prepare_amplitudes,
    qfunc,
)
from examples.synthesis.algorithms.amplitude_amplification_and_estimation.oblivious_amplitude_amplification.operations import (
    encode,
    reflect,
    zero_phase,
)


@qfunc
def amplify(
    repetitions: int,
    amplitudes: list[float],
    data: Output[QArray[QBit]],
    block: Output[QArray[QBit]],
) -> None:
    allocate(2, block)
    allocate(2, data)
    inplace_prepare_amplitudes(amplitudes, 0, data)
    encode(block, data)
    for i in range(repetitions):
        zero_phase(block)
        reflect(block, data)


## First experiment: block encoding

Read the joint data/block distribution from actual local CPU execution. Filter on the named block output rather than decoding physical bitstrings. The exact probabilities replace the cloud sampler here; no finite-shot or hardware claim is made.


In [ ]:
program_before = synth.synthesize(
    amplify,
    classical_inputs={"repetitions": 0, "amplitudes": list(amplitudes)},
    max_workspace=0,
)
result_before = fq.run(program_before)
rows_before = tuple(
    sorted(
        ((data, block), p)
        for (data, block), p in program_before.probabilities(
            result_before, registers=("data", "block")
        ).items()
    )
)
before = Stage(
    program_before, rows_before, sum(p for (_, block), p in rows_before if block == 0)
)
print("Good data/probability rows:", before.good_rows())
print("Fraction of good states:", before.good_probability)
assert np.isclose(before.good_probability, 0.25, rtol=0, atol=1e-12)


## Second experiment: one amplification iteration

Reinitialize the identical seed-one input and apply exactly one iteration. Preserve both logical outputs and inspect actual resources.


In [ ]:
program_after = synth.synthesize(
    amplify,
    classical_inputs={"repetitions": 1, "amplitudes": list(amplitudes)},
    max_workspace=0,
)
result_after = fq.run(program_after)
rows_after = tuple(
    sorted(
        ((data, block), p)
        for (data, block), p in program_after.probabilities(
            result_after, registers=("data", "block")
        ).items()
    )
)
after = Stage(
    program_after, rows_after, sum(p for (_, block), p in rows_after if block == 0)
)
print("Good data/probability rows:", after.good_rows())
print("Fraction of good states:", after.good_probability)
print("Before resources:", program_before.report["resources"])
print("After resources:", program_after.report["resources"])


## Check the good block separately from normalization

The source's final cell sums the complete table, so its final assertion checks normalization rather than the success probability. Correct that check explicitly: sum only rows where `block == 0`, and verify normalization separately. The robust near-unitary case remains background; this workflow validates the concrete unitary example.


In [ ]:
assert np.isclose(after.good_probability, 1.0, rtol=0, atol=1e-12)
assert np.isclose(sum(p for _, p in before.probabilities), 1.0, rtol=0, atol=1e-12)
assert np.isclose(sum(p for _, p in after.probabilities), 1.0, rtol=0, atol=1e-12)
print("Good-block probabilities:", before.good_probability, after.good_probability)
